In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, when
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Tugas6") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [6]:
# Extract membaca dari tiga sumber
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# Data transaksi
print("Data Transaksi")
print("Data transaksi:", df_transaksi.count(), "baris")
df_transaksi.printSchema()

# Data produk
print("Data Produk")
print("Data produk:", df_produk.count(), "baris")
df_produk.printSchema()

# Data ulasan
print("Data Ulasan")
print("Data ulasan:", df_ulasan.count(), "baris")
df_ulasan.printSchema()

Data Transaksi
Data transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data Produk
Data produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data Ulasan
Data ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [30]:
# Join transaksi dengan ulasan
gabungan_etl = df_transaksi.join(
    df_ulasan,
    on="order_id",
    how="left"
)

# Join dengan produk
gabungan_etl = gabungan_etl.join(
    df_produk,
    on="product_id",
    how="inner"
)

# Menghitung total pendapatan
gabungan_etl = gabungan_etl.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

gabungan_etl.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

In [35]:
# Membuat kolom ada_rating sebelum mengisi data kosong
gabungan_etl = gabungan_etl.withColumn(
    "ada_rating",
    col("rating").isNotNull()
)

# Mengisi data kosong dengan 0
gabungan_etl = gabungan_etl.na.fill({
    "rating": 0
})

# Menampilkan hasil
gabungan_etl.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_rating|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|      true|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|      true|
|        10|     TX5|   

In [32]:
# Membuat folder tujuan di HDFS
!hdfs dfs -mkdir -p /user/indra/tugas6

# Menyimpan hasil ETL ke HDFS dalam format Parquet
gabungan_etl.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(
        "hdfs://localhost:9000/user/indra/tugas6/hasil_etl"
    )

print("Hasil ETL berhasil disimpan ke HDFS!")

Hasil ETL berhasil disimpan ke HDFS!


In [33]:
!hdfs dfs -ls -R /user/indra/tugas6/hasil_etl

df_final = spark.read.parquet(
    "hdfs://localhost:9000/user/indra/tugas6/hasil_etl"
)

print("Jumlah baris hasil akhir:", df_final.count())

-rw-r--r--   3 indra supergroup          0 2026-09-30 22:23 /user/indra/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - indra supergroup          0 2026-09-30 22:23 /user/indra/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 indra supergroup      16366 2026-09-30 22:23 /user/indra/tugas6/hasil_etl/kategori=Elektronik/part-00000-af0ca44c-6a4a-438c-881f-b65a0927a50a.c000.snappy.parquet
drwxr-xr-x   - indra supergroup          0 2026-09-30 22:23 /user/indra/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 indra supergroup      11133 2026-09-30 22:23 /user/indra/tugas6/hasil_etl/kategori=Fashion/part-00000-af0ca44c-6a4a-438c-881f-b65a0927a50a.c000.snappy.parquet
drwxr-xr-x   - indra supergroup          0 2026-09-30 22:23 /user/indra/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 indra supergroup      10392 2026-09-30 22:23 /user/indra/tugas6/hasil_etl/kategori=Kesehatan/part-00000-af0ca44c-6a4a-438c-881f-b65a0927a50a.c000.snappy.parquet
drwxr-xr-x   - indra supergroup          0 2026-09-3

In [34]:
from pyspark.sql.functions import col, count, sum, when, round

# Menghitung jumlah transaksi dan transaksi yang memiliki ulasan
insight = df_final.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    sum(
        when(col("ada_rating") == True, 1)
        .otherwise(0)
    ).alias("transaksi_dengan_ulasan")
)

# Menghitung persentase transaksi dengan ulasan
insight = insight.withColumn(
    "persentase_ulasan",
    round(
        col("transaksi_dengan_ulasan") /
        col("total_transaksi") * 100,
        2
    )
)

# Menampilkan hasil dari persentase terendah
insight.orderBy(
    col("persentase_ulasan").asc()
).show()

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|     Makanan|            536|                    369|            68.84|
|   Kesehatan|            961|                    662|            68.89|
|     Fashion|           1035|                    726|            70.14|
|Rumah Tangga|            815|                    575|            70.55|
|  Elektronik|           1653|                   1168|            70.66|
+------------+---------------+-----------------------+-----------------+

